# Music Annotation Project V7.9.1: Safety-First Confirmation and Resumable Stage 4

**Notebook 22. Runtime:** CPU. Select **Runtime > Run all**.

This self-contained continuation does not dynamically import the previous notebook and does not repeat coarse or fine discovery. It:

1. verifies the saved full-development V7.8 equivalence result;
2. loads the completed checkpoint tables;
3. confirms only missing candidate/page pairs for three distinct route families;
4. performs full reconstruction for frozen V7.8 and those three candidates only;
5. writes every page atomically with hashes and resumes after interruption;
6. defers contact-sheet rendering until all page metrics are durable.

Notebook 09 is not read or modified. Frozen V7.8 independent data is prohibited.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy psutil


## Configuration


In [ ]:
import cv2, json, hashlib, gc, os, re, psutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V79_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9'
SOURCE_RUN=None  # Set explicitly to the crashed Notebook 22 run if automatic discovery selects the wrong run.
OUTPUT_ROOT=ROOT/'observed_ink_preservation_challenger_v7_9_1'
MAX_PAGES=None
FORCE_CONFIRMATION=False
FORCE_STAGE4=False
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4

SELECTED_CANDIDATES=[
 'v79_connected_dark_d80_w1.75_a12_100',
 'v79_union_d100_w4.0_a12_1800',
 'v79_consensus_d100_w1.75_a18_100',
]
PROHIBITED=['independent_structural_validation_v7_8_builder','independent_structural_validation_v7_8_rebuilt','independent_policy_validation_v7_8']

def assert_development_path(path):
 s=str(Path(path)).lower()
 if any(x in s for x in PROHIBITED):raise RuntimeError(f'Prohibited independent V7.8 path: {path}')
def latest_run(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
if SOURCE_RUN is None:
 SOURCE_RUN=latest_run(V79_ROOT,['tables/baseline_equivalence.csv','tables/fine_discovery_route_metrics_checkpoint.csv','tables/all_page_confirmation_route_metrics_checkpoint.csv','tables/development_page_roles.csv'])
if SOURCE_RUN is None:raise RuntimeError('Set SOURCE_RUN to the crashed Notebook 22 run.')
SOURCE_RUN=Path(SOURCE_RUN);assert_development_path(SOURCE_RUN)
V75=latest_run(V75_ROOT,['tables/cache_status.csv'])
if V75 is None:raise RuntimeError('No complete V7.5 controlled development run found.')
assert_development_path(V75)
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_safety_first_resumable';OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables';PAGE_RESULTS=OUT/'stage4_pages';FINALISTS=OUT/'finalists';AUDIT=OUT/'audit'
for p in [OUT,TABLES,PAGE_RESULTS,FINALISTS,AUDIT]:p.mkdir(parents=True,exist_ok=False)
print('Source run:',SOURCE_RUN);print('V7.5:',V75);print('Output:',OUT)


## Self-contained frozen baseline and V7.9 route engine


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):
 assert_development_path(folder);return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def loadz(path):
 assert_development_path(path)
 with np.load(path) as z:return {k:z[k] for k in z.files}
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def atomic_csv(df,path):
 tmp=Path(str(path)+'.tmp');df.to_csv(tmp,index=False);tmp.replace(path)
def atomic_json(value,path):
 tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,default=str));tmp.replace(path)
def rss_gb():return psutil.Process(os.getpid()).memory_info().rss/(1024**3)
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0]
 assert np.array_equal(out[mask==0],original[mask==0]);return out
def background_surface(observed,mask,sigma=45.0):
 visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
 num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
 return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps_v78(observed,mask):
 dark=(observed<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=stats[i]
  if 3<=a<=650 and w<=40 and hg<=40:compact[lab==i]=True
 k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
 return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(compact.astype(np.uint8),k)>0
def page_invariants(observed,mask,telea,clean):
 paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps_v78(observed,mask);edges=cv2.Canny(observed,70,170)>0
 clean_ink=cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0;clean_edges=cv2.Canny(clean,70,170)>0;m=mask>0
 base=m&(horizontal|vertical|compact)&(observed<200);remove=m&~base&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD);amb=m&~base&~remove
 return {'paper':paper,'normalised':normalised,'horizontal':horizontal,'vertical':vertical,'compact':compact,'edges':edges,'clean_ink':clean_ink,'clean_edges':clean_edges,'base_preserve':base,'base_remove':remove,'base_ambiguous':amb,'telea':telea}
def render(observed,mask,inv,preserve,remove,ambiguous):
 out=observed.copy();out[remove]=inv['paper'][remove];out[ambiguous]=inv['telea'][ambiguous];out[preserve]=observed[preserve];return immutable(observed,out,mask)
def boundary_band(mask,width=3):
 m=(mask>0).astype(np.uint8);inner=cv2.erode(m,np.ones((2*width+1,2*width+1),np.uint8))>0;return (m>0)&~inner
def parse_candidate(name):
 m=re.fullmatch(r'v79_(connected_dark|edge_stroke|consensus|union)_d(\d+)_w([0-9.]+)_a(\d+)_(\d+)',name)
 if not m:raise ValueError(name)
 return m.group(1),{'dark_threshold':int(m.group(2)),'stroke_width_limit':float(m.group(3)),'min_area':int(m.group(4)),'max_area':int(m.group(5))}
def connected_map(dark,boundary,min_area,max_area):
 n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);out=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,h,a=stats[i];comp=lab==i
  if min_area<=a<=max_area and w<=80 and h<=80 and np.any(comp&boundary):out[comp]=True
 return out
def glyph_support(observed,mask,inv,mode,p):
 dark=(observed<p['dark_threshold'])&(mask>0);connected=connected_map(dark,boundary_band(mask,3),p['min_area'],p['max_area']);width=cv2.distanceTransform(dark.astype(np.uint8),cv2.DIST_L2,5)
 edge=dark&(cv2.dilate(inv['edges'].astype(np.uint8),np.ones((3,3),np.uint8))>0)&(width<=p['stroke_width_limit']);raw={'connected_dark':connected,'edge_stroke':edge,'consensus':connected&edge,'union':connected|edge}[mode]
 n,lab,stats,_=cv2.connectedComponentsWithStats(raw.astype(np.uint8),8);out=np.zeros_like(raw)
 for i in range(1,n):
  if stats[i,cv2.CC_STAT_AREA]>=p['min_area']:out[lab==i]=True
 return out
def candidate_routes(observed,mask,inv,candidate=None):
 glyph=np.zeros_like(mask,dtype=bool)
 if candidate is not None:
  mode,p=parse_candidate(candidate);glyph=glyph_support(observed,mask,inv,mode,p)
 m=mask>0;preserve=m&(inv['horizontal']|inv['vertical']|inv['compact']|glyph)&(observed<200);remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(inv['normalised']>=GEOMETRY_THRESHOLD);amb=m&~preserve&~remove
 return preserve,remove,amb,glyph
def route_metrics(inv,glyph):
 added=glyph&~inv['base_preserve'];tp=int(np.sum(added&inv['clean_ink']));fp=int(np.sum(added&~inv['clean_ink']))
 return {'added_pixels':tp+fp,'added_true_print':tp,'added_false_annotation':fp,'added_precision':tp/max(tp+fp,1),'clean_print_recall_gain':tp/max(int(inv['clean_ink'].sum()),1)}


## Load saved checkpoints and verify the baseline gate


In [ ]:
def required(name):
 p=SOURCE_RUN/'tables'/name
 if not p.exists():raise RuntimeError(f'Missing required checkpoint: {p}')
 return p
baseline=pd.read_csv(required('baseline_equivalence.csv'))
for col in ['geometry_exact','outside_immutable']:
 baseline[col]=baseline[col].astype(str).str.lower().isin(['true','1','yes'])
if not baseline.geometry_exact.all() or not baseline.outside_immutable.all() or not (baseline.different_pixels==0).all():raise RuntimeError('Saved frozen V7.8 equivalence gate did not pass.')
roles=pd.read_csv(required('development_page_roles.csv'));pages=roles.page.astype(str).tolist()
if MAX_PAGES is not None:pages=pages[:MAX_PAGES]
confirmation=pd.read_csv(required('all_page_confirmation_route_metrics_checkpoint.csv'))
status=pd.read_csv(V75/'tables'/'cache_status.csv');cache_lookup={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean');observed_map=image_map(ROOT/'aligned')
missing=[p for p in pages if p not in clean_map or p not in observed_map or p not in cache_lookup]
if missing:raise RuntimeError(f'Development page inputs missing: {missing}')
print('Baseline PASS. Pages:',len(pages),'Existing confirmation rows:',len(confirmation))


## Incremental all-page confirmation for the selected distinct candidates


In [ ]:
CONFIRM=TABLES/'all_page_confirmation_route_metrics_checkpoint.csv'
base=confirmation.copy()
if CONFIRM.exists() and not FORCE_CONFIRMATION:base=pd.concat([base,pd.read_csv(CONFIRM)],ignore_index=True)
base=base.drop_duplicates(['page','candidate'],keep='last')
for i,page in enumerate(pages,1):
 missing=[x for x in SELECTED_CANDIDATES if FORCE_CONFIRMATION or not ((base.page.astype(str)==page)&(base.candidate.astype(str)==x)).any()]
 if not missing:continue
 z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean);rows=[]
 for candidate in missing:
  _,_,_,glyph=candidate_routes(observed,mask,inv,candidate);rows.append({'stage':'all_page_confirmation','page':page,'candidate':candidate,**route_metrics(inv,glyph)});del glyph
 base=pd.concat([base,pd.DataFrame(rows)],ignore_index=True).drop_duplicates(['page','candidate'],keep='last');atomic_csv(base,CONFIRM)
 print('Confirmation',i,'/',len(pages),page,'new rows',len(rows),'RSS',round(rss_gb(),3),'GB');del z,clean,observed,mask,inv,rows;gc.collect()
selected=base[base.candidate.isin(SELECTED_CANDIDATES)].copy();counts=selected.groupby('candidate').page.nunique()
if any(counts.get(x,0)!=len(pages) for x in SELECTED_CANDIDATES):raise RuntimeError(f'Incomplete confirmation: {counts.to_dict()}')
summary=selected.groupby('candidate',as_index=False).agg(pages=('page','nunique'),added_true_print=('added_true_print','sum'),added_false_annotation=('added_false_annotation','sum'),mean_precision=('added_precision','mean'),median_precision=('added_precision','median'),mean_recall_gain=('clean_print_recall_gain','mean'),pages_with_gain=('added_true_print',lambda x:int((x>0).sum())),pages_with_false_preservation=('added_false_annotation',lambda x:int((x>0).sum())))
summary['global_precision']=summary.added_true_print/(summary.added_true_print+summary.added_false_annotation).clip(lower=1);summary['false_per_true']=summary.added_false_annotation/summary.added_true_print.clip(lower=1)
summary.to_csv(TABLES/'all_page_route_summary_safety_first.csv',index=False);display(summary)


## Resumable page-atomic Stage 4

Every page writes its candidate arrays, page metrics, component metrics and hashed completion marker before memory is released. Contact sheets are not generated here.


In [ ]:
def valid_complete(page):
 p=PAGE_RESULTS/f'{page}_complete.json'
 if FORCE_STAGE4 or not p.exists():return False
 try:
  m=json.loads(p.read_text())
  return m.get('candidates')==SELECTED_CANDIDATES and all((OUT/r).exists() and sha256_file(OUT/r)==h for r,h in m.get('outputs',{}).items())
 except:return False
def tolerant(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.sum():return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pd=cv2.dilate(pred.astype(np.uint8),k)>0;p=float(np.sum(pred&rd)/max(pred.sum(),1));r=float(np.sum(ref&pd)/ref.sum());return p,r,float(2*p*r/max(p+r,1e-12))
def full_metrics(clean,recon,observed,mask,inv,preserve,remove,glyph):
 m=mask>0;ink=cv2.adaptiveThreshold(recon,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0;edge=cv2.Canny(recon,70,170)>0;ip,ir,if1=tolerant(inv['clean_ink'],ink,m);ep,er,ef1=tolerant(inv['clean_edges'],edge,m);d=np.abs(clean.astype(np.float32)-recon.astype(np.float32));tp=int(np.sum(preserve&inv['clean_ink']));fp=int(np.sum(preserve&~inv['clean_ink']))
 return {'mae':float(d[m].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'true_print_preserved':tp,'false_annotation_preserved':fp,'preserve_precision':tp/max(tp+fp,1),'glyph_pixels':int(glyph.sum()),'removed_clean_print_fraction':float(np.sum(remove&inv['clean_ink']&m)/max(m.sum(),1)),'outside_mask_drift':int(np.sum(recon[~m]!=observed[~m]))}
for index,page in enumerate(pages,1):
 if valid_complete(page):print('Resume skip',page);continue
 z=loadz(cache_lookup[page]);clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0);mask=z['mask'];inv=page_invariants(observed,mask,z['telea'],clean);n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);page_rows=[];component_rows=[];hashes={}
 for candidate in ['v78_baseline']+SELECTED_CANDIDATES:
  preserve,remove,amb,glyph=candidate_routes(observed,mask,inv,None if candidate=='v78_baseline' else candidate);recon=render(observed,mask,inv,preserve,remove,amb);q=full_metrics(clean,recon,observed,mask,inv,preserve,remove,glyph)
  if q['outside_mask_drift']!=0:raise RuntimeError(f'{page} {candidate}: outside-mask drift')
  page_rows.append({'page':page,'candidate':candidate,**q});npz=FINALISTS/f'{page}_{candidate}.npz';np.savez_compressed(npz,reconstruction=recon,preserve=preserve.astype(np.uint8),remove=remove.astype(np.uint8),ambiguous=amb.astype(np.uint8),glyph=glyph.astype(np.uint8));hashes[str(npz.relative_to(OUT))]=sha256_file(npz)
  recon_ink=cv2.adaptiveThreshold(recon,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0;recon_edges=cv2.Canny(recon,70,170)>0;diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32))
  for cid in range(1,n):
   x,y,w,h,a=[int(v) for v in stats[cid]]
   if a<MIN_COMPONENT_AREA:continue
   comp=labels[y:y+h,x:x+w]==cid;ip,ir,if1=tolerant(inv['clean_ink'][y:y+h,x:x+w],recon_ink[y:y+h,x:x+w],comp);ep,er,ef1=tolerant(inv['clean_edges'][y:y+h,x:x+w],recon_edges[y:y+h,x:x+w],comp)
   component_rows.append({'page':page,'component':cid,'candidate':candidate,'x':x,'y':y,'w':w,'h':h,'area':a,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'mae':float(diff[y:y+h,x:x+w][comp].mean()),'false_annotation_preserved':int(np.sum(preserve[y:y+h,x:x+w]&comp&~inv['clean_ink'][y:y+h,x:x+w]))})
  del preserve,remove,amb,glyph,recon,recon_ink,recon_edges,diff;gc.collect()
 pm=PAGE_RESULTS/f'{page}_metrics.csv';cm=PAGE_RESULTS/f'{page}_components.csv';atomic_csv(pd.DataFrame(page_rows),pm);atomic_csv(pd.DataFrame(component_rows),cm);hashes[str(pm.relative_to(OUT))]=sha256_file(pm);hashes[str(cm.relative_to(OUT))]=sha256_file(cm);atomic_json({'page':page,'candidates':SELECTED_CANDIDATES,'completed_utc':datetime.now(timezone.utc).isoformat(),'outputs':hashes},PAGE_RESULTS/f'{page}_complete.json')
 print('Stage 4',index,'/',len(pages),page,'RSS',round(rss_gb(),3),'GB');del z,clean,observed,mask,inv,labels,stats,page_rows,component_rows;gc.collect()


## Merge results and create a bounded review queue


In [ ]:
incomplete=[p for p in pages if not valid_complete(p)]
if incomplete:raise RuntimeError(f'Incomplete Stage 4 pages: {incomplete}')
page_df=pd.concat([pd.read_csv(PAGE_RESULTS/f'{p}_metrics.csv') for p in pages],ignore_index=True);component_df=pd.concat([pd.read_csv(PAGE_RESULTS/f'{p}_components.csv') for p in pages],ignore_index=True)
page_df.to_csv(TABLES/'stage4_page_metrics.csv',index=False);component_df.to_csv(TABLES/'stage4_component_metrics.csv',index=False)
final=page_df.groupby('candidate',as_index=False).agg(pages=('page','nunique'),ink_recall=('ink_recall','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),mae=('mae','mean'),false_annotation_preserved=('false_annotation_preserved','sum'),true_print_preserved=('true_print_preserved','sum'),preserve_precision=('preserve_precision','mean'),outside_mask_drift=('outside_mask_drift','max'))
base=final[final.candidate=='v78_baseline'].iloc[0]
for x in ['ink_recall','ink_f1','edge_f1','mae','false_annotation_preserved','true_print_preserved']:final['delta_'+x]=final[x]-base[x]
final.to_csv(TABLES/'stage4_finalist_summary.csv',index=False)
queue=[]
for candidate,g in page_df[page_df.candidate!='v78_baseline'].groupby('candidate'):
 for label,col,asc in [('largest_false_preservation','false_annotation_preserved',False),('largest_true_print_gain','true_print_preserved',False),('lowest_ink_recall','ink_recall',True)]:
  for rank,(_,r) in enumerate(g.sort_values(col,ascending=asc).head(12).iterrows(),1):queue.append({'category':label,'rank':rank,'candidate':candidate,'page':r.page,'value':r[col]})
pd.DataFrame(queue).to_csv(TABLES/'diagnostic_review_queue.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.9.1-self-contained-resumable','source_run':str(SOURCE_RUN),'v7_5_reference':str(V75),'selected_candidates':SELECTED_CANDIDATES,'pages':len(pages),'baseline_equivalence_passed':True,'all_page_confirmation_complete':True,'stage4_complete':True,'notebook_09_modified':False,'v7_8_independent_data_read':False,'promotion_authorised':False}
atomic_json(manifest,OUT/'run_manifest.json');display(final);print('COMPLETE:',OUT)
